PARCIAL FINAL 
Luisa Fernanda Baquero

In [0]:
# primero creamos el volumen y subimos los archicos 
catalog = "workspace"  
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.electiva")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.electiva.datos_csv")

In [0]:
# verifico que quedaron los 8 archivos csv
display(dbutils.fs.ls("/Volumes/workspace/electiva/datos_csv"))

In [0]:
from pyspark.sql import functions as F

catalog = "workspace"     
schema  = "electiva"
volume  = "datos_csv"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

vol_path = f"/Volumes/{catalog}/{schema}/{volume}"

archivos = {
    "customers.csv":            "customers",
    "employees.csv":            "employee",
    "orders.csv":               "orders",
    "productcategories.csv":    "product_categories",
    "productsubcategories.csv": "product_sub_categories",
    "products.csv":             "products",
    "vendors.csv":              "vendor",
    "vendorproduct.csv":        "vendor_product",
}

for archivo, tabla in archivos.items():
    df = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .option("encoding", "ISO-8859-1")
          .option("nullValue", "NULL")
          .csv(f"{vol_path}/{archivo}"))

    if tabla == "orders":
        for c in ["OrderDate", "DueDate", "ShipDate"]:
            df = df.withColumn(c, F.coalesce(
                F.try_to_timestamp(F.col(c).cast("string"), F.lit("M/d/yyyy")),
                F.try_to_timestamp(F.col(c).cast("string"))
            ).cast("date"))

    (df.write.mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{catalog}.{schema}.{tabla}"))
    print(f"{tabla}: {df.count()} filas")

 1. primer punto cadena user_name

In [0]:
%sql
SELECT
  customerid,
  fullname,
  concat(customerid, '-',
         upper(substring(firstname, 1, 2)), '-',
         lower(substring(lastname, 1, 2)), '-AW') AS user_name
FROM customers
ORDER BY lastname, firstname;

2. Segundo punto crear la tabla page_user

In [0]:
%sql
CREATE OR REPLACE TABLE page_user (customerid INT, user_name  STRING) USING DELTA;

se ponen los datos

In [0]:
%sql
INSERT INTO page_user
SELECT customerid, concat(customerid, '-', upper(substring(firstname, 1, 2)), '-', lower(substring(lastname, 1, 2)), '-AW') AS user_name
FROM customers;

In [0]:
%sql
SELECT * FROM page_user ORDER BY customerid;

3. Tercera parte tabla nueva con todas las orders realizadas en el mes de agosto de 2011

In [0]:
%sql
CREATE OR REPLACE VIEW v_utl_ordenes_mitad_anio AS
WITH base AS (
  SELECT
    p.productname,
    p.modelname,
    u.user_name,
    o.salesorderid,
    c.categoryname,
    o.orderdate,
    o.linetotal
  FROM orders o
  JOIN products p               ON o.productid     = p.productid
  JOIN product_sub_categories s ON p.subcategoryid = s.subcategoryid
  JOIN product_categories c     ON s.categoryid    = c.categoryid
  JOIN page_user u              ON o.customerid    = u.customerid
  WHERE o.orderdate >= DATE'2011-08-01'
    AND o.orderdate <  DATE'2011-09-01'
),
con_descuento AS (
  SELECT *,
    CASE
      WHEN categoryname ILIKE 'Bikes'       THEN 0.10
      WHEN categoryname ILIKE 'Clothing'    THEN 0.15
      WHEN categoryname ILIKE 'Components'  THEN 0.08
      WHEN categoryname ILIKE 'Accessories' THEN 0.20
      ELSE 0
    END AS `descuento_mitad_año`
  FROM base
)
SELECT *,
       linetotal - linetotal * `descuento_mitad_año` AS total_descuentos_aplicados
FROM con_descuento;

In [0]:
%sql
SELECT * FROM ordenes_mitad_anio
ORDER BY user_name, salesorderid;

In [0]:
%sql
CREATE OR REPLACE TABLE ordenes_mitad_anio AS
SELECT * FROM ordenes_mitad_anio;

4. Cuarta parte crear una vista materializada en el esquema, llamada customer_class que contenga el
user_name de todos los clientes del año 2011 , una columna user_level donde indique si el
cliente es “Oro”, “Plata” o “Ocasional”, además debe contener para cada cliente su
customerid, su customer fullname y la cantidad de órdenes que realizó en el año

In [0]:
%sql
CREATE OR REPLACE MATERIALIZED VIEW customer_class AS
SELECT
  COUNT(DISTINCT o.salesorderid) AS ammount_of_orders,
  u.user_name,
  c.fullname                     AS customerfullname,
  CASE
    WHEN COUNT(DISTINCT o.salesorderid) >= 3 THEN 'Oro'
    WHEN COUNT(DISTINCT o.salesorderid) = 2  THEN 'Plata'
    ELSE 'Ocasional'
  END                            AS user_level,
  c.customerid
FROM orders o
JOIN customers c ON o.customerid = c.customerid
JOIN page_user u ON o.customerid = u.customerid
WHERE year(o.orderdate) = 2011
GROUP BY c.customerid, c.fullname, u.user_name;

In [0]:
%sql
SELECT * FROM customer_class;

In [0]:
%sql
SELECT user_level, COUNT(*) AS clientes
FROM customer_class
GROUP BY user_level;

Conclusión

En este parcial cargué los datos en Databricks y resolví los cuatro puntos: el user_name de los clientes, la tabla page_user, la vista con los descuentos de agosto y la vista materializada customer_class. Me sirvió para repasar lo que vimos en el curso, como los joins, el CASE WHEN y las vistas.
Dos aclaraciones: en el Punto 3 usé la fórmula escrita en el enunciado, y en el Punto 4 mi vista muestra las 205 filas completas, porque el ejemplo tenía activado el límite de 100.

Profe, muchas gracias por el semestre,prendí bastante en esta electiva y me llevo cosas que me van a servir mucho. Gracias por la paciencia.